# Shockwave Submission Notebook

**Stage:** Shockwave  
**Plan:** `docs/plans/shockwave.md`  
**Primary output:** `submission_shockwave.csv`

Runs the leak-aware feature pipeline, trains the diverse base models, compares RRF/z-score ensembles on the locked validation split, and can retrain on 100% of training data for Kaggle submission.

In [ ]:
from pathlib import Path
import sys
import gc

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

try:
    import catboost as cb
except Exception as exc:
    cb = None
    print(f"CatBoost unavailable: {exc}")

from src.utils import load_train, load_test, train_val_split, compute_ndcg_fast, make_submission
from src.modeling import get_lgb_device_params
from src.final_pipeline import (
    COMP_RAW,
    build_test_feature_frame,
    build_validation_feature_frames,
    get_feature_cols,
    make_relevance,
    minmax_normalize_per_search,
    optimize_blend_weights,
    rrf_score,
    save_run_json,
    sort_for_group_model,
    z_normalize_per_search,
)

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
OUT_DIR = REPO_ROOT / "outputs"
MODEL_DIR = REPO_ROOT / "models"
OUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
print(REPO_ROOT)

/home/akshitanchan/Projects/hotel-trivago-but-its-expedia


## Configuration

For fast verification, set `USE_SAMPLE=True`. For the final validation and submission, keep `USE_SAMPLE=False`. If this machine has no CUDA device, set `CATBOOST_TASK_TYPE="CPU"`.

In [2]:
RANDOM_STATE = 42
USE_SAMPLE = False
SAMPLE_FRAC = 0.15
SVD_COMPONENTS = 20

TRY_LGB_GPU = True
RUN_RANK_XENDCG = True
RUN_CATBOOST = True
CATBOOST_TASK_TYPE = "GPU"  # switch to "CPU" if CUDA is unavailable
CATBOOST_KEEP_SVD_FACTORS = False  # keeps svd_affinity_score, drops 40 factor columns for CatBoost memory
CATBOOST_USE_ID_CATEGORICALS = True  # disable if CatBoost memory blows up

RUN_FINAL_SUBMISSION = False
USE_TRAIN_TEST_COUNTS = False  # True is transductive but target-free; False uses train-only counts

PW_NUM_BOOST_ROUND = 8000
PW_EARLY_STOPPING = 120
LR_NUM_BOOST_ROUND = 10000
LR_EARLY_STOPPING = 150
XENDCG_NUM_BOOST_ROUND = 6000
XENDCG_EARLY_STOPPING = 150
REG_NUM_BOOST_ROUND = 5000
REG_EARLY_STOPPING = 80

LGB_DEVICE_PARAMS = get_lgb_device_params(verbose=True) if TRY_LGB_GPU else {}
print("LightGBM device params:", LGB_DEVICE_PARAMS)

GPU mode enabled - using LightGBM GPU acceleration
LightGBM device params: {'device': 'gpu', 'gpu_use_dp': False}


In [3]:
def load_run_json(name):
    path = OUT_DIR / name
    if not path.exists():
        return {}
    with path.open() as f:
        return json.load(f)


def sanitize_lgb_params(params, objective=None, metric=None):
    params = dict(params)
    if objective is not None:
        params["objective"] = objective
    if metric is not None:
        params["metric"] = metric
    params.pop("num_iterations", None)
    params.pop("early_stopping_round", None)
    params.pop("early_stopping_rounds", None)
    if LGB_DEVICE_PARAMS:
        params.update(LGB_DEVICE_PARAMS)
    else:
        params.pop("device", None)
        params.pop("gpu_use_dp", None)
    params.update({"verbose": -1, "seed": RANDOM_STATE, "num_threads": -1})
    return params


pw_run = load_run_json("pointwise_run2.json")
lr_run = load_run_json("lambdarank_run2.json")

PW_PARAMS = sanitize_lgb_params(
    pw_run.get("best_params", {
        "objective": "binary",
        "metric": "binary_logloss",
        "num_leaves": 511,
        "learning_rate": 0.01,
        "min_child_samples": 200,
        "lambda_l1": 0.1,
        "lambda_l2": 1.0,
        "min_gain_to_split": 0.01,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
    }),
    objective="binary",
    metric="binary_logloss",
)

LR_PARAMS = sanitize_lgb_params(
    lr_run.get("best_params", {
        "objective": "lambdarank",
        "metric": "ndcg",
        "ndcg_eval_at": [5],
        "num_leaves": 511,
        "learning_rate": 0.01,
        "min_child_samples": 200,
        "lambda_l1": 0.1,
        "lambda_l2": 1.0,
        "min_gain_to_split": 0.01,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
    }),
    objective="lambdarank",
    metric="ndcg",
)
LR_PARAMS["ndcg_eval_at"] = [5]
LR_PARAMS["label_gain"] = [0, 1, 2, 3, 4, 5]  # match local linear 5/1/0 NDCG, not default exponential gains

XENDCG_PARAMS = dict(LR_PARAMS)
XENDCG_PARAMS["objective"] = "rank_xendcg"
XENDCG_PARAMS["metric"] = "ndcg"

REG_PARAMS = sanitize_lgb_params(
    {
        "objective": "huber",
        "alpha": 0.9,
        "metric": "mae",
        "num_leaves": 127,
        "learning_rate": 0.05,
        "min_child_samples": 200,
        "lambda_l1": 0.1,
        "lambda_l2": 1.0,
        "feature_fraction": 0.6,
        "bagging_fraction": 0.7,
        "bagging_freq": 5,
    }
)

print("Pointwise params:", PW_PARAMS)
print("LambdaRank params:", LR_PARAMS)
print("RankXENDCG params:", XENDCG_PARAMS)
print("Regression params:", REG_PARAMS)

Pointwise params: {'objective': 'binary', 'metric': 'binary_logloss', 'num_leaves': 511, 'learning_rate': 0.01, 'min_child_samples': 200, 'lambda_l1': 0.1, 'lambda_l2': 1.0, 'min_gain_to_split': 0.01, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'feature_pre_filter': False, 'device': 'gpu', 'gpu_use_dp': False, 'verbose': -1, 'seed': 42, 'num_threads': -1}
LambdaRank params: {'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [5], 'num_leaves': 511, 'learning_rate': 0.01, 'min_child_samples': 200, 'lambda_l1': 0.1, 'lambda_l2': 1.0, 'min_gain_to_split': 0.01, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'feature_pre_filter': False, 'device': 'gpu', 'gpu_use_dp': False, 'verbose': -1, 'seed': 42, 'num_threads': -1, 'label_gain': [0, 1, 2, 3, 4, 5]}
RankXENDCG params: {'objective': 'rank_xendcg', 'metric': 'ndcg', 'ndcg_eval_at': [5], 'num_leaves': 511, 'learning_rate': 0.01, 'min_child_samples': 200, 'lambda_l1': 0.1, 'lambda_l2'

## Build Locked Validation Features

Target-derived maps are fitted only from `train_raw`. The pipeline preserves the old missing-aware `prop_location_score2_*` relative features and adds imputed `prop_location_score2_imp_*` variants.

In [4]:
raw_df = load_train(sample_frac=SAMPLE_FRAC if USE_SAMPLE else None, random_state=RANDOM_STATE)
raw_rows = len(raw_df)
raw_searches = raw_df["srch_id"].nunique()
train_raw, val_raw = train_val_split(raw_df, random_state=RANDOM_STATE)
count_reference = train_raw  # strict validation: fit count/rank maps on train fold only

print(f"Raw rows: {raw_rows:,}  searches: {raw_searches:,}")
print(f"Train rows: {len(train_raw):,}  Val rows: {len(val_raw):,}")
del raw_df
gc.collect()

train_df, val_df = build_validation_feature_frames(
    train_raw,
    val_raw,
    count_reference_frame=count_reference,
    svd_components=SVD_COMPONENTS,
)
del train_raw, val_raw, count_reference
gc.collect()

assert train_df.duplicated(["srch_id", "prop_id"]).sum() == 0
assert val_df.duplicated(["srch_id", "prop_id"]).sum() == 0

feature_cols = get_feature_cols(train_df)
missing_in_val = sorted(set(feature_cols) - set(val_df.columns))
assert not missing_in_val, missing_in_val[:10]

nan_counts = val_df[feature_cols].isna().sum().sort_values(ascending=False)
print(f"Features: {len(feature_cols)}")
print("Top NaN feature counts:")
print(nan_counts[nan_counts > 0].head(25))

Raw rows: 4,958,347  searches: 199,795
Train rows: 3,966,682  Val rows: 991,665
Features: 172
Top NaN feature counts:
visitor_hist_starrating             942474
star_diff_visitor                   942474
price_vs_visitor_hist               942240
visitor_hist_adr_usd                942240
visitor_hist_price_diff_rank        942240
visitor_hist_price_diff             942240
srch_query_affinity_score           927296
srch_query_affinity_score_rank      927296
comp_max_diff                       678394
comp_avg_diff                       678394
distance_price_interaction          325095
orig_destination_distance           325095
log_distance                        325095
prop_location_score2_diff_mean      219870
prop_location_score2_rank           219870
prop_location_score2_diff_median    219870
prop_location_score2_std             37651
prop_location_score2_min             18346
prop_location_score2_max             18346
prop_review_score_diff_mean           1480
prop_review_score     

## Train Base Models On Validation Split

In [5]:
pred_df = val_df[["srch_id", "prop_id", "booking_bool", "click_bool"]].copy()

# Pointwise LightGBM anchor
pw_train = lgb.Dataset(train_df[feature_cols], label=train_df["booking_bool"].to_numpy())
pw_val = lgb.Dataset(val_df[feature_cols], label=val_df["booking_bool"].to_numpy(), reference=pw_train)
pw_model = lgb.train(
    PW_PARAMS,
    pw_train,
    valid_sets=[pw_val],
    num_boost_round=PW_NUM_BOOST_ROUND,
    callbacks=[lgb.early_stopping(PW_EARLY_STOPPING), lgb.log_evaluation(200)],
)
pred_df["pw_score"] = pw_model.predict(val_df[feature_cols], num_iteration=pw_model.best_iteration)
pw_ndcg = compute_ndcg_fast(pred_df, "pw_score")
print(f"Pointwise NDCG@5: {pw_ndcg:.6f}  trees={pw_model.best_iteration}")

Training until validation scores don't improve for 120 rounds
[200]	valid_0's binary_logloss: 0.111349
[400]	valid_0's binary_logloss: 0.10958
[600]	valid_0's binary_logloss: 0.108806
[800]	valid_0's binary_logloss: 0.108387
[1000]	valid_0's binary_logloss: 0.108104
[1200]	valid_0's binary_logloss: 0.107925
[1400]	valid_0's binary_logloss: 0.107795
[1600]	valid_0's binary_logloss: 0.107707
[1800]	valid_0's binary_logloss: 0.107642
[2000]	valid_0's binary_logloss: 0.107588
[2200]	valid_0's binary_logloss: 0.107551
[2400]	valid_0's binary_logloss: 0.10753
[2600]	valid_0's binary_logloss: 0.107509
[2800]	valid_0's binary_logloss: 0.107498
[3000]	valid_0's binary_logloss: 0.107483
Early stopping, best iteration is:
[3065]	valid_0's binary_logloss: 0.107482
Pointwise NDCG@5: 0.402220  trees=3065


In [6]:
# LambdaRank LightGBM, explicitly sorted by srch_id before group sizes are built
lr_train_sorted, X_train_lr, y_train_lr, group_train = sort_for_group_model(
    train_df, feature_cols, make_relevance(train_df)
)
lr_val_sorted, X_val_lr, y_val_lr, group_val = sort_for_group_model(
    val_df, feature_cols, make_relevance(val_df)
)
lr_train = lgb.Dataset(X_train_lr, label=y_train_lr, group=group_train)
lr_val = lgb.Dataset(X_val_lr, label=y_val_lr, group=group_val, reference=lr_train)
lr_model = lgb.train(
    LR_PARAMS,
    lr_train,
    valid_sets=[lr_val],
    num_boost_round=LR_NUM_BOOST_ROUND,
    callbacks=[lgb.early_stopping(LR_EARLY_STOPPING), lgb.log_evaluation(200)],
)
lr_pred = lr_val_sorted[["srch_id", "prop_id"]].copy()
lr_pred["lr_score"] = lr_model.predict(X_val_lr, num_iteration=lr_model.best_iteration)
pred_df = pred_df.merge(lr_pred, on=["srch_id", "prop_id"], how="left")
lr_ndcg = compute_ndcg_fast(pred_df, "lr_score")
print(f"LambdaRank NDCG@5: {lr_ndcg:.6f}  trees={lr_model.best_iteration}")

Training until validation scores don't improve for 150 rounds
[200]	valid_0's ndcg@5: 0.377368
[400]	valid_0's ndcg@5: 0.38586
[600]	valid_0's ndcg@5: 0.390002
[800]	valid_0's ndcg@5: 0.393207
[1000]	valid_0's ndcg@5: 0.394575
[1200]	valid_0's ndcg@5: 0.396513
[1400]	valid_0's ndcg@5: 0.397282
[1600]	valid_0's ndcg@5: 0.397282
[1800]	valid_0's ndcg@5: 0.398143
Early stopping, best iteration is:
[1791]	valid_0's ndcg@5: 0.398278
LambdaRank NDCG@5: 0.398278  trees=1791


In [7]:
# Optional RankXENDCG: same grouped data, different ranking objective with linear label_gain
xendcg_model = None
if RUN_RANK_XENDCG:
    xendcg_train = lgb.Dataset(X_train_lr, label=y_train_lr, group=group_train)
    xendcg_val = lgb.Dataset(X_val_lr, label=y_val_lr, group=group_val, reference=xendcg_train)
    xendcg_model = lgb.train(
        XENDCG_PARAMS,
        xendcg_train,
        valid_sets=[xendcg_val],
        num_boost_round=XENDCG_NUM_BOOST_ROUND,
        callbacks=[lgb.early_stopping(XENDCG_EARLY_STOPPING), lgb.log_evaluation(200)],
    )
    xendcg_pred = lr_val_sorted[["srch_id", "prop_id"]].copy()
    xendcg_pred["xendcg_score"] = xendcg_model.predict(X_val_lr, num_iteration=xendcg_model.best_iteration)
    pred_df = pred_df.merge(xendcg_pred, on=["srch_id", "prop_id"], how="left")
    xendcg_ndcg = compute_ndcg_fast(pred_df, "xendcg_score")
    print(f"RankXENDCG NDCG@5: {xendcg_ndcg:.6f}  trees={xendcg_model.best_iteration}")
else:
    xendcg_ndcg = None
    print("Skipping RankXENDCG")

Training until validation scores don't improve for 150 rounds
[200]	valid_0's ndcg@5: 0.379992
[400]	valid_0's ndcg@5: 0.389262
[600]	valid_0's ndcg@5: 0.394209
[800]	valid_0's ndcg@5: 0.39688
[1000]	valid_0's ndcg@5: 0.398903
[1200]	valid_0's ndcg@5: 0.400373
[1400]	valid_0's ndcg@5: 0.401332
[1600]	valid_0's ndcg@5: 0.402415
[1800]	valid_0's ndcg@5: 0.402741
[2000]	valid_0's ndcg@5: 0.403656
[2200]	valid_0's ndcg@5: 0.404173
[2400]	valid_0's ndcg@5: 0.404731
Early stopping, best iteration is:
[2399]	valid_0's ndcg@5: 0.404792
RankXENDCG NDCG@5: 0.404792  trees=2399


In [8]:
# Graded relevance regressor: captures click relevance that pointwise booking ignores
y_train_reg = make_relevance(train_df).astype("float32")
y_val_reg = make_relevance(val_df).astype("float32")
reg_weight = np.where(y_train_reg == 5, 5.0, np.where(y_train_reg == 1, 3.0, 1.0)).astype("float32")
reg_train = lgb.Dataset(train_df[feature_cols], label=y_train_reg, weight=reg_weight)
reg_val = lgb.Dataset(val_df[feature_cols], label=y_val_reg, reference=reg_train)
reg_model = lgb.train(
    REG_PARAMS,
    reg_train,
    valid_sets=[reg_val],
    num_boost_round=REG_NUM_BOOST_ROUND,
    callbacks=[lgb.early_stopping(REG_EARLY_STOPPING), lgb.log_evaluation(200)],
)
pred_df["reg_score"] = reg_model.predict(val_df[feature_cols], num_iteration=reg_model.best_iteration)
reg_ndcg = compute_ndcg_fast(pred_df, "reg_score")
print(f"Regression NDCG@5: {reg_ndcg:.6f}  trees={reg_model.best_iteration}")

Training until validation scores don't improve for 80 rounds
[200]	valid_0's l1: 0.302627
Early stopping, best iteration is:
[133]	valid_0's l1: 0.300364
Regression NDCG@5: 0.367433  trees=133


In [9]:
cat_model = None
cb_ndcg = None

if RUN_CATBOOST and cb is not None:
    cb_feature_cols = get_feature_cols(train_df, keep_svd_factors=CATBOOST_KEEP_SVD_FACTORS)
    cat_features_names = [
        "site_id",
        "visitor_location_country_id",
        "prop_country_id",
        "prop_starrating",
        "prop_brand_bool",
        "promotion_flag",
        "srch_saturday_night_bool",
        "month",
        "day_of_week",
    ]
    if CATBOOST_USE_ID_CATEGORICALS:
        cat_features_names += ["prop_id", "srch_destination_id"]
    cat_features_idx = [i for i, c in enumerate(cb_feature_cols) if c in cat_features_names]
    missing_cats = set(cat_features_names) - set(cb_feature_cols)
    assert not missing_cats, f"Missing CatBoost categorical columns: {missing_cats}"
    assert len(cat_features_idx) == len(cat_features_names)
    cb_train = train_df.sort_values(["srch_id", "prop_id"], kind="mergesort").reset_index(drop=True)
    cb_val = val_df.sort_values(["srch_id", "prop_id"], kind="mergesort").reset_index(drop=True)
    cb_y_train = make_relevance(cb_train)
    cb_y_val = make_relevance(cb_val)

    train_pool = cb.Pool(
        cb_train[cb_feature_cols],
        label=cb_y_train,
        group_id=cb_train["srch_id"].to_numpy(),
        cat_features=cat_features_idx,
    )
    val_pool = cb.Pool(
        cb_val[cb_feature_cols],
        label=cb_y_val,
        group_id=cb_val["srch_id"].to_numpy(),
        cat_features=cat_features_idx,
    )
    cb_params = {
        "loss_function": "YetiRank",
        "eval_metric": "NDCG:top=5;type=Base",
        "iterations": 3000,
        "learning_rate": 0.03,
        "depth": 8,
        "l2_leaf_reg": 3.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.8,
        "border_count": 254,
        "task_type": CATBOOST_TASK_TYPE,
        "random_seed": RANDOM_STATE,
        "early_stopping_rounds": 100,
        "allow_writing_files": False,
        "verbose": 200,
    }
    cat_model = cb.CatBoost(cb_params)
    cat_model.fit(train_pool, eval_set=val_pool)
    cb_pred = cb_val[["srch_id", "prop_id"]].copy()
    cb_pred["cb_score"] = cat_model.predict(val_pool)
    pred_df = pred_df.merge(cb_pred, on=["srch_id", "prop_id"], how="left")
    cb_ndcg = compute_ndcg_fast(pred_df, "cb_score")
    print(f"CatBoost NDCG@5: {cb_ndcg:.6f}  trees={cat_model.get_best_iteration()}")
else:
    print("Skipping CatBoost")

Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.2891954	best: 0.2891954 (0)	total: 1.01s	remaining: 50m 40s
200:	test: 0.3551797	best: 0.3551908 (199)	total: 3m 17s	remaining: 45m 50s
400:	test: 0.3663551	best: 0.3663551 (400)	total: 6m 24s	remaining: 41m 30s
600:	test: 0.3719121	best: 0.3720380 (598)	total: 9m 26s	remaining: 37m 39s
800:	test: 0.3755932	best: 0.3755932 (800)	total: 12m 24s	remaining: 34m 3s
1000:	test: 0.3781256	best: 0.3781256 (1000)	total: 15m 21s	remaining: 30m 39s
1200:	test: 0.3805761	best: 0.3805761 (1200)	total: 18m 16s	remaining: 27m 22s
1400:	test: 0.3821908	best: 0.3822353 (1395)	total: 21m 10s	remaining: 24m 9s
1600:	test: 0.3836168	best: 0.3836239 (1579)	total: 24m 4s	remaining: 21m 2s
1800:	test: 0.3848754	best: 0.3848834 (1799)	total: 26m 58s	remaining: 17m 57s
2000:	test: 0.3859666	best: 0.3859733 (1999)	total: 29m 50s	remaining: 14m 54s
2200:	test: 0.3870900	best: 0.3870900 (2200)	total: 32m 44s	remaining: 11m 53s
2400:	test: 0.3881135	best: 0.3881135 (2400)	total: 35m 38s	remaining: 8m 5

## Ensemble Selection

In [ ]:
score_cols = [c for c in ["pw_score", "lr_score", "xendcg_score", "reg_score", "cb_score"] if c in pred_df.columns]
base_scores = {col: compute_ndcg_fast(pred_df, col) for col in score_cols}
print("Base scores:")
for col, score in base_scores.items():
    print(f"  {col}: {score:.6f}")

for col in score_cols:
    pred_df[f"{col}_mm"] = minmax_normalize_per_search(pred_df, col)
    pred_df[f"{col}_z"] = z_normalize_per_search(pred_df, col)

pred_df["rrf_score"] = rrf_score(pred_df, score_cols, k=60)
rrf_ndcg = compute_ndcg_fast(pred_df, "rrf_score")
print(f"RRF NDCG@5: {rrf_ndcg:.6f}")

z_cols = [f"{col}_z" for col in score_cols]
weights, opt_ndcg = optimize_blend_weights(pred_df, z_cols)
pred_df["opt_z_blend"] = sum(w * pred_df[col] for w, col in zip(weights, z_cols))
print("Optimized z-blend:", opt_ndcg, dict(zip(score_cols, weights.round(4))))

mm_cols = [f"{col}_mm" for col in score_cols]
mm_weights, mm_ndcg = optimize_blend_weights(pred_df, mm_cols)
pred_df["opt_mm_blend"] = sum(w * pred_df[col] for w, col in zip(mm_weights, mm_cols))
print("Optimized minmax blend:", mm_ndcg, dict(zip(score_cols, mm_weights.round(4))))

ensemble_candidates = {
    "rrf_score": rrf_ndcg,
    "opt_z_blend": opt_ndcg,
    "opt_mm_blend": mm_ndcg,
}
best_ensemble_col = max(ensemble_candidates, key=ensemble_candidates.get)
best_ensemble_ndcg = ensemble_candidates[best_ensemble_col]
print(f"Best ensemble: {best_ensemble_col}  NDCG@5={best_ensemble_ndcg:.6f}")

pred_path = OUT_DIR / "shockwave_validation_predictions.parquet"
pred_df.to_parquet(pred_path, index=False)
save_run_json(
    OUT_DIR / "shockwave_validation_run.json",
    {
        "use_sample": USE_SAMPLE,
        "sample_frac": SAMPLE_FRAC if USE_SAMPLE else None,
        "n_train": int(len(train_df)),
        "n_val": int(len(val_df)),
        "n_features": int(len(feature_cols)),
        "base_scores": {k: float(v) for k, v in base_scores.items()},
        "rrf_ndcg5": float(rrf_ndcg),
        "opt_z_ndcg5": float(opt_ndcg),
        "opt_z_weights": dict(zip(score_cols, [float(w) for w in weights])),
        "opt_mm_ndcg5": float(mm_ndcg),
        "opt_mm_weights": dict(zip(score_cols, [float(w) for w in mm_weights])),
        "best_ensemble_col": best_ensemble_col,
        "best_ensemble_ndcg5": float(best_ensemble_ndcg),
        "best_iterations": {
            "pointwise": int(pw_model.best_iteration),
            "lambdarank": int(lr_model.best_iteration),
            "rank_xendcg": None if xendcg_model is None else int(xendcg_model.best_iteration),
            "regression": int(reg_model.best_iteration),
            "catboost": None if cat_model is None else int(cat_model.get_best_iteration()),
        },
    },
)
print(f"Saved {pred_path}")

Base scores:
  pw_score: 0.402220
  lr_score: 0.398278
  xendcg_score: 0.404792
  reg_score: 0.367433
  cb_score: 0.390199
RRF NDCG@5: 0.401669
Optimized z-blend: 0.4062593197862218 {'pw_score': np.float64(0.251), 'lr_score': np.float64(0.2513), 'xendcg_score': np.float64(0.296), 'reg_score': np.float64(0.0014), 'cb_score': np.float64(0.2004)}
Optimized minmax blend: 0.40634523037239806 {'pw_score': np.float64(0.2811), 'lr_score': np.float64(0.2157), 'xendcg_score': np.float64(0.2809), 'reg_score': np.float64(0.0039), 'cb_score': np.float64(0.2183)}
Best ensemble: opt_mm_blend  NDCG@5=0.406345
Saved /home/akshitanchan/Projects/hotel-trivago-but-its-expedia/outputs/shockwave_validation_predictions.parquet


## Optional Final Submission Retrain

Set `RUN_FINAL_SUBMISSION=True` only after validation selects the final model set and ensemble.

In [12]:
if not RUN_FINAL_SUBMISSION:
    full_train_raw = load_train()
    test_raw = load_test()
    if USE_TRAIN_TEST_COUNTS:
        count_reference = pd.concat(
            [
                full_train_raw[["prop_id", "srch_destination_id", "prop_location_score2"]],
                test_raw[["prop_id", "srch_destination_id", "prop_location_score2"]],
            ],
            ignore_index=True,
        )
    else:
        count_reference = full_train_raw

    final_train_df, test_df = build_test_feature_frame(
        full_train_raw,
        test_raw,
        count_reference_frame=count_reference,
        svd_components=SVD_COMPONENTS,
    )
    final_feature_cols = get_feature_cols(final_train_df)
    assert set(final_feature_cols).issubset(test_df.columns)

    final_scores = test_df[["srch_id", "prop_id"]].copy()
    full_scores_for_norm = final_scores.copy()

    final_pw = lgb.train(
        PW_PARAMS,
        lgb.Dataset(final_train_df[final_feature_cols], label=final_train_df["booking_bool"].to_numpy()),
        num_boost_round=int(pw_model.best_iteration),
    )
    final_scores["pw_score"] = final_pw.predict(test_df[final_feature_cols])

    tr_sorted, X_full_lr, y_full_lr, group_full = sort_for_group_model(
        final_train_df, final_feature_cols, make_relevance(final_train_df)
    )
    test_lr_sorted = test_df.sort_values(["srch_id", "prop_id"], kind="mergesort").reset_index(drop=True)
    final_lr = lgb.train(
        LR_PARAMS,
        lgb.Dataset(X_full_lr, label=y_full_lr, group=group_full),
        num_boost_round=int(lr_model.best_iteration),
    )
    lr_test_pred = test_lr_sorted[["srch_id", "prop_id"]].copy()
    lr_test_pred["lr_score"] = final_lr.predict(test_lr_sorted[final_feature_cols])
    final_scores = final_scores.merge(lr_test_pred, on=["srch_id", "prop_id"], how="left")

    if xendcg_model is not None and "xendcg_score" in score_cols:
        final_xendcg = lgb.train(
            XENDCG_PARAMS,
            lgb.Dataset(X_full_lr, label=y_full_lr, group=group_full),
            num_boost_round=int(xendcg_model.best_iteration),
        )
        xendcg_test_pred = test_lr_sorted[["srch_id", "prop_id"]].copy()
        xendcg_test_pred["xendcg_score"] = final_xendcg.predict(test_lr_sorted[final_feature_cols])
        final_scores = final_scores.merge(xendcg_test_pred, on=["srch_id", "prop_id"], how="left")

    y_full_reg = make_relevance(final_train_df).astype("float32")
    reg_weight = np.where(y_full_reg == 5, 5.0, np.where(y_full_reg == 1, 3.0, 1.0)).astype("float32")
    final_reg = lgb.train(
        REG_PARAMS,
        lgb.Dataset(final_train_df[final_feature_cols], label=y_full_reg, weight=reg_weight),
        num_boost_round=int(reg_model.best_iteration),
    )
    final_scores["reg_score"] = final_reg.predict(test_df[final_feature_cols])

    if cat_model is not None and "cb_score" in score_cols:
        cb_feature_cols = get_feature_cols(final_train_df, keep_svd_factors=CATBOOST_KEEP_SVD_FACTORS)
        cat_features_names = [
            "site_id", "visitor_location_country_id", "prop_country_id",
            "prop_starrating", "prop_brand_bool", "promotion_flag",
            "srch_saturday_night_bool", "month", "day_of_week",
        ]
        if CATBOOST_USE_ID_CATEGORICALS:
            cat_features_names += ["prop_id", "srch_destination_id"]
        cat_features_idx = [i for i, c in enumerate(cb_feature_cols) if c in cat_features_names]
        missing_cats = set(cat_features_names) - set(cb_feature_cols)
        assert not missing_cats, f"Missing CatBoost categorical columns: {missing_cats}"
        assert len(cat_features_idx) == len(cat_features_names)
        cb_full = final_train_df.sort_values(["srch_id", "prop_id"], kind="mergesort").reset_index(drop=True)
        cb_test = test_df.sort_values(["srch_id", "prop_id"], kind="mergesort").reset_index(drop=True)
        full_pool = cb.Pool(
            cb_full[cb_feature_cols],
            label=make_relevance(cb_full),
            group_id=cb_full["srch_id"].to_numpy(),
            cat_features=cat_features_idx,
        )
        test_pool = cb.Pool(
            cb_test[cb_feature_cols],
            group_id=cb_test["srch_id"].to_numpy(),
            cat_features=cat_features_idx,
        )
        final_cb_params = dict(cb_params)
        final_cb_params.update(cat_model.get_params())
        assert final_cb_params.get("loss_function") == "YetiRank", final_cb_params
        final_cb_params.pop("early_stopping_rounds", None)
        final_cb_params["iterations"] = max(1, int(cat_model.get_best_iteration()) + 1)
        final_cb = cb.CatBoost(final_cb_params)
        final_cb.fit(full_pool)
        cb_test_pred = cb_test[["srch_id", "prop_id"]].copy()
        cb_test_pred["cb_score"] = final_cb.predict(test_pool)
        final_scores = final_scores.merge(cb_test_pred, on=["srch_id", "prop_id"], how="left")

    final_score_cols = [c for c in score_cols if c in final_scores.columns]
    if best_ensemble_col == "rrf_score":
        final_scores["final_score"] = rrf_score(final_scores, final_score_cols, k=60)
    elif best_ensemble_col == "opt_z_blend":
        for col in final_score_cols:
            final_scores[f"{col}_z"] = z_normalize_per_search(final_scores, col)
        final_scores["final_score"] = sum(
            w * final_scores[f"{col}_z"] for w, col in zip(weights, final_score_cols)
        )
    else:
        for col in final_score_cols:
            final_scores[f"{col}_mm"] = minmax_normalize_per_search(final_scores, col)
        final_scores["final_score"] = sum(
            w * final_scores[f"{col}_mm"] for w, col in zip(mm_weights, final_score_cols)
        )

    sub_path = REPO_ROOT / "submission_shockwave.csv"
    sub = make_submission(final_scores, "final_score", sub_path)
    assert len(sub) == len(test_raw)
    assert sub.duplicated().sum() == 0
    assert set(sub["srch_id"]) == set(test_raw["srch_id"])
    print(f"Saved {sub_path} with {len(sub):,} rows")
else:
    print("RUN_FINAL_SUBMISSION=False; validation-only run complete.")

Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	total: 1.23s	remaining: 1h 1m 34s
200:	total: 4m 7s	remaining: 57m 17s
400:	total: 7m 59s	remaining: 51m 44s
600:	total: 11m 46s	remaining: 46m 53s
800:	total: 15m 27s	remaining: 42m 20s
1000:	total: 19m 7s	remaining: 38m 5s
1200:	total: 22m 45s	remaining: 33m 59s
1400:	total: 26m 21s	remaining: 29m 59s
1600:	total: 29m 57s	remaining: 26m 5s
1800:	total: 33m 33s	remaining: 22m 15s
2000:	total: 37m 9s	remaining: 18m 27s
2200:	total: 40m 44s	remaining: 14m 41s
2400:	total: 44m 20s	remaining: 10m 58s
2600:	total: 47m 59s	remaining: 7m 16s
2800:	total: 51m 37s	remaining: 3m 34s
2994:	total: 55m 7s	remaining: 0us
Saved /home/akshitanchan/Projects/hotel-trivago-but-its-expedia/submission_shockwave.csv with 4,959,183 rows
